In [3]:
# this script reads and evaluates features related to pc's and then loads a wrf-file

import pandas as pd # pandas is more labeled tabular oriented. works well if headers are provided in files.
import numpy as np
import matplotlib.pyplot as plt
import wrf
import xarray
from netCDF4 import Dataset

import cartopy.crs as crs
from cartopy.feature import NaturalEarthFeature

# plotting features
from cartopy import crs
from cartopy.feature import NaturalEarthFeature, COLORS
from matplotlib.cm import get_cmap

# also have a look in xarray for netcdf-supported python language and scripting

# also check https://geocat.ucar.edu/pages/software.html for 
# 1) geocat software
# 2) PyNIO: NCL input output library. Likely very useful.
# 3) PyNGL: Graphics library. Probably less useful as matplotlib can do the same.
# 4) wrf-python: 
# 5) MetPy: meteorology-oriented python package

In [4]:
# colormap definition with white in the middle
n=39
x = 0.5
lower = plt.cm.seismic_r(np.linspace(0, x, n))
white = plt.cm.seismic_r(np.ones(80-2*n)*x)
upper = plt.cm.seismic_r(np.linspace(1-x, 1, n))
colors = np.vstack((lower, white, upper))
from matplotlib.colors import LinearSegmentedColormap
tmap = LinearSegmentedColormap.from_list('map_white', colors)
# tmap = matplotlib.colors.LinearSegmentedColormap.from_list('map_white', colors)

In [5]:
def pbl_height_thv_plus_k(thv_col, z_col, thv_crit=1.5):
    """
    Estimate PBL height using theta-v + K method (as in MATLAB pblHeights_SD_thVplusK).
    thv_col : 1-D array of virtual potential temperature (K)
    z_col   : 1-D array of height above ground level (m)
    thv_crit: threshold in K above minimum theta-v (default 1.5 K)
    Returns PBL height in metres, or NaN if not found.
    """
    # Find minimum theta-v and its index
    ind_min = np.nanargmin(thv_col)
    val_min = thv_col[ind_min]

    # Search upward from the minimum for first level exceeding val_min + thv_crit
    above = np.where(thv_col[ind_min:] > val_min + thv_crit)[0]
    if len(above) == 0:
        return np.nan

    ind_cross = above[0] + ind_min   # absolute index in full column

    # Linear interpolation between ind_cross-1 and ind_cross (mirrors MATLAB)
    thv_lo = thv_col[ind_cross - 1]
    thv_hi = thv_col[ind_cross]
    z_lo   = z_col[ind_cross - 1]
    z_hi   = z_col[ind_cross]

    zi = ((val_min + thv_crit - thv_lo) / (thv_hi - thv_lo)) * (z_hi - z_lo) + z_lo
    return zi

In [6]:
"""
Combined cross-section plot: 6 panels per time step
  Row 1 (a, b) : 1 km WRF
  Row 2 (c, d) : 111 m smoothed-1km WRF
  Row 3 (e, f) : 111 m WRF
Left column  : V-component + theta + Fr-transition lines
Right column : W-component + theta + Fr-transition lines
Shared colorbars below each column.

v2: PBL top for all three simulations from the thv + 1.5 K method
    (minimum thv + 1.5 K, linearly interpolated), instead of
    WRF PBLH (1 km) and near-surface thv + 0.5 K (111 m).
"""

import numpy as np
import matplotlib
import matplotlib.colors as mcolors
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from netCDF4 import Dataset
import wrf

# ============================================================
# ── USER SETTINGS ───────────────────────────────────────────
# ============================================================

# ── 1 km ────────────────────────────────────────────────────
baseFolder_1km  = '/glade/campaign/univ/ucsb0023/duine/wrf/wrf460/sundowners/SWEX2022/IOP10/'
run_1km         = 'run_1km_545_24_z107_2way/'
fName_1km       = baseFolder_1km + run_1km + 'wrfout_d03_2022-05-11_18:00:00'
idt_range       = range(138, 283, 1)          # time indices to loop over
target_lat_1km  = [34.440168, 34.440168]

# ── 111 m smoothed ──────────────────────────────────────────
baseFolder_sm   = '/glade/campaign/univ/ucsb0023/duine/wrf/wrf460/sundowners/SWEX2022/IOP10/run_1km_545_24_z107_2way/ndown_111m_km_opt2_2Dom_smth1km/'
target_lat_sm   = [34.440168, 34.440168]

# ── 111 m (no smooth) ───────────────────────────────────────
baseFolder_111  = '/glade/campaign/univ/ucsb0023/duine/wrf/wrf460/sundowners/SWEX2022/IOP10/run_1km_545_24_z107_2way/ndown_111m_km_opt2_2Dom/'
target_lat_111  = [34.440168, 34.440168]

# ── Shared 111 m file list ───────────────────────────────────
wrffi  = 'wrfout_d04_2022-05-'
tfiles = ['12_17','12_18','12_19','12_20','12_21',
          '12_22','12_23','13_00','13_01','13_02','13_03',
          '13_04','13_05','13_06','13_07','13_08','13_09',
          '13_10','13_11','13_12','13_13','13_14','13_15',
          '13_16','13_17']

# ── Cross-section line ───────────────────────────────────────
xlim1, xlim2 = 34.36, 34.60
ylim1, ylim2 = 0, 3000
lonLocs = [-119.6593, -119.6593]
stNames = ['UWOW', 'UWOW']

# ── Contour levels ───────────────────────────────────────────
V_levels  = np.arange(-24, 26,  2)
W_levels  = np.arange(-5,   5.2, 0.2)
th_levels = np.arange(260, 340,  1)

g = 9.81   # m s-2

# ── Colormap ─────────────────────────────────────────────────
try:
    tmap
except NameError:
    tmap = 'RdBu_r'   # fallback if tmap is not already defined

# ============================================================
# ── HELPER : PBL height from virtual-theta profile ──────────
# ============================================================
def pbl_height_thv_plus_k(thv_col, z_agl_col, thv_crit=1.5):
    """
    theta-v + K method (as in MATLAB pblHeights_SD_thVplusK):
    first level above the minimum thv where thv exceeds that minimum
    by thv_crit K, linearly interpolated.  Returns height AGL (m),
    or NaN if not found.
    """
    if np.all(np.isnan(thv_col)):
        return np.nan
    ind_min = np.nanargmin(thv_col)
    val_min = thv_col[ind_min]
    above = np.where(thv_col[ind_min:] > val_min + thv_crit)[0]
    if len(above) == 0:
        return np.nan
    ind_cross = above[0] + ind_min   # absolute index in full column
    thv_lo = thv_col[ind_cross - 1]
    thv_hi = thv_col[ind_cross]
    z_lo   = z_agl_col[ind_cross - 1]
    z_hi   = z_agl_col[ind_cross]
    return float(((val_min + thv_crit - thv_lo) / (thv_hi - thv_lo)) * (z_hi - z_lo) + z_lo)


# ============================================================
# ── HELPER : compute Froude-number profile ───────────────────
# ============================================================
def compute_froude(z_np, thv_np, V_np, pblh_np, offset):
    nz, nx = thv_np.shape
    fr = np.full(nx, np.nan)
    for ix in range(nx):
        z_col   = z_np[:, ix]
        thv_col = thv_np[:, ix]
        V_col   = V_np[:, ix]
        zi      = pblh_np[ix]
        if zi <= 0:
            zi = 1.0
        above = np.where(z_col > zi)[0]
        if len(above) == 0:
            continue
        ind_zi   = max(above[0] - 1, 0)
        mean_thv = np.nanmean(thv_col[: ind_zi + 1])
        ind_free = min(ind_zi + offset, nz - 1)
        thv_free = thv_col[ind_free]
        mean_V   = np.nanmean(V_col[: ind_zi + 1])
        g_prime  = g * (thv_free - mean_thv) / mean_thv
        if g_prime <= 0:
            continue
        fr[ix] = np.abs(mean_V) / np.sqrt(g_prime * zi)
    return fr


# ============================================================
# ── HELPER : find sub↔super transitions ─────────────────────
# ============================================================
def find_transitions(fr_profile, lats_1d, lat_limit=34.518):
    is_super = fr_profile > 1.0
    trans    = {'sub_to_super': None, 'super_to_sub': None}
    nx       = len(fr_profile)
    for ix in range(nx - 2, -1, -1):   # north → south
        if np.isnan(fr_profile[ix]) or np.isnan(fr_profile[ix + 1]):
            continue
        if lats_1d[ix] > lat_limit:
            continue
        fr_a,  fr_b  = fr_profile[ix],  fr_profile[ix + 1]
        lat_a, lat_b = lats_1d[ix],     lats_1d[ix + 1]
        lat_cross    = lat_a + (1.0 - fr_a) * (lat_b - lat_a) / (fr_b - fr_a)
        if is_super[ix] and not is_super[ix + 1]:
            if trans['sub_to_super'] is None:
                trans['sub_to_super'] = lat_cross
        elif not is_super[ix] and is_super[ix + 1]:
            if trans['sub_to_super'] is not None and trans['super_to_sub'] is None:
                trans['super_to_sub'] = lat_cross
    return trans


# ============================================================
# ── HELPER : extract cross-section data from an ncfile ───────
# ============================================================
def get_cross_data(ncfile, tidx, latLocs, use_wrf_pblh=False):
    """
    Returns dict with numpy arrays ready for plotting + Froude computation.
    use_wrf_pblh : if True use WRF PBLH variable;
                   if False compute from thv profile (thv + 1.5 K, used for all runs in v2).
    """
    latLocs_cross = [latLocs[0], latLocs[1]]
    xy = wrf.ll_to_xy(ncfile, latLocs_cross, lonLocs)

    ter  = wrf.getvar(ncfile, "ter", -1)
    lat  = wrf.getvar(ncfile, "lat", -1)

    timeWRF = wrf.getvar(ncfile, "Times", tidx)
    z   = wrf.getvar(ncfile, "z",       tidx)
    V   = wrf.getvar(ncfile, "va",      tidx)
    W   = wrf.getvar(ncfile, "wa",      tidx)
    th  = wrf.getvar(ncfile, "th",      tidx)
    qv  = wrf.getvar(ncfile, "QVAPOR",  tidx)
    thv = th * (1.0 + 0.61 * qv)

    nz_full = z.shape[0]
    lats    = np.tile(wrf.to_np(lat), (nz_full, 1, 1))

    cross_start = wrf.CoordPair(lat=latLocs_cross[0], lon=lonLocs[0])
    cross_end   = wrf.CoordPair(lat=latLocs_cross[1], lon=lonLocs[1])
    ter_line    = wrf.interpline(ter, wrfin=ncfile,
                                 start_point=cross_start, end_point=cross_end)

    xy_line    = wrf.xy(V, start_point=xy[:, 0], end_point=xy[:, 1])
    V_cross    = wrf.interp2dxy(V,   xy_line, meta=True)
    W_cross    = wrf.interp2dxy(W,   xy_line, meta=True)
    z_cross    = wrf.interp2dxy(z,   xy_line, meta=True)
    th_cross   = wrf.interp2dxy(th,  xy_line, meta=True)
    thv_cross  = wrf.interp2dxy(thv, xy_line, meta=True)
    lats_cross = wrf.interp2dxy(lats, xy_line, meta=True)

    V_np    = wrf.to_np(V_cross)
    z_np    = wrf.to_np(z_cross)
    thv_np  = wrf.to_np(thv_cross)
    lats_np = wrf.to_np(lats_cross)
    ter_np  = wrf.to_np(ter_line)

    nz, nx = thv_np.shape

    if use_wrf_pblh:
        pblh_raw   = wrf.getvar(ncfile, "PBLH", tidx)
        pblh_cross = wrf.interpline(pblh_raw, wrfin=ncfile,
                                    start_point=cross_start, end_point=cross_end)
        pblh_np = wrf.to_np(pblh_cross)
    else:
        z_agl_np = z_np - ter_np[np.newaxis, :]
        pblh_np  = np.array([pbl_height_thv_plus_k(thv_np[:, ix], z_agl_np[:, ix])
                              for ix in range(nx)])

    if   nz == 54:  offset = 2
    elif nz == 80:  offset = 3
    elif nz == 106: offset = 4
    else:           offset = 2

    fr_profile = compute_froude(z_np, thv_np, V_np, pblh_np, offset)

    # PDT time string
    ts      = pd.to_datetime(wrf.to_np(timeWRF))
    tsPDT   = ts - pd.DateOffset(hours=7)
    tPDTstr = tsPDT.strftime('%Y%m%d_%H%M')

    return dict(
        V_cross=V_cross, W_cross=W_cross, z_cross=z_cross,
        th_cross=th_cross, lats_cross=lats_cross,
        ter_line=ter_line, lats_np=lats_np,
        fr_profile=fr_profile,
        tPDTstr=tPDTstr,
    )


# ============================================================
# ── HELPER : draw one cross-section panel ───────────────────
# ============================================================
def draw_panel(ax, lats_cross, z_cross, field_cross, th_cross,
               ter_line, levels, cmap,
               target_latitudes,
               trans,
               show_ylabel=False, show_xlabel=False,
               label_letter='', time_str='',
               show_legend=False, show_fr_lines=True):

    cf = ax.contourf(wrf.to_np(lats_cross), wrf.to_np(z_cross),
                     wrf.to_np(field_cross),
                     levels=levels, cmap=cmap, extend='both')

    ct = ax.contour(wrf.to_np(lats_cross), wrf.to_np(z_cross),
                    wrf.to_np(th_cross),
                    levels=th_levels, colors='k', linewidths=1)

    ax.fill_between(wrf.to_np(lats_cross[0, :]),
                    np.zeros(len(ter_line)),
                    wrf.to_np(ter_line),
                    facecolor='saddlebrown')

    plt.clabel(ct, inline=1, fontsize=14, levels=th_levels[::3], fmt='%1.0f')

    ax.scatter(target_latitudes, [50, 50], marker='^',
               color='yellow', edgecolor='black', linewidth=1.5, s=300)

    if show_fr_lines:
        if trans['sub_to_super'] is not None:
            ax.axvline(x=trans['sub_to_super'], color='limegreen', linewidth=3.5,
                       linestyle='--',
                       label='sub to super' if show_legend else '_nolegend_')
        if trans['super_to_sub'] is not None:
            ax.axvline(x=trans['super_to_sub'], color='deeppink', linewidth=3.5,
                       linestyle='--',
                       label='super to sub' if show_legend else '_nolegend_')
        if show_legend and any(trans.values()):
            ax.legend(fontsize=13, loc='upper right')

    ax.set_xlim(xlim1, xlim2)
    ax.set_ylim(ylim1, ylim2)

    if show_ylabel:
        ax.set_ylabel('Elevation [m]', fontsize=22)
    else:
        ax.set_yticklabels([])

    if show_xlabel:
        ax.set_xlabel('Latitude [°N]', fontsize=22)
    else:
        ax.set_xticklabels([])

    ax.tick_params(labelsize=18)

    # panel letter + time label (top-left)
    ax.text(xlim1 + 0.003, ylim2 - 220,
            f'({label_letter}) {time_str} PDT',
            fontsize=18,
            bbox=dict(facecolor='white', edgecolor='white',
                      boxstyle='round,pad=0.2'),
            zorder=10)

    return cf   # return for colorbar


# ============================================================
# ── OPEN 1 km FILE once (it stays open for all time steps) ───
# ============================================================
print('Opening 1 km file …')
ncfile_1km = Dataset(fName_1km)

# Precompute latLocs for 1km (same for all time steps)
latLocs_1km = [xlim1, xlim2]


# ============================================================
# ── BUILD A TIME-INDEX MAP for 111 m files ───────────────────
# Each entry maps global time index → (file_index, local_t)
# 25 files × 6 time steps = 150 entries (indices 0..149)
# ============================================================
time_map = []   # list of (fi, local_t)
for fi in range(25):
    for t in range(6):
        time_map.append((fi, t))


# ── keep track of open 111 m file to avoid re-opening ────────
current_fi_sm  = -1
current_fi_111 = -1
ncfile_sm      = None
ncfile_111     = None


# ============================================================
# ── MAIN LOOP over 1 km time indices ─────────────────────────
# ============================================================
for global_idt, idt in enumerate(idt_range):

    # ── 1 km data ────────────────────────────────────────────
    print(f'[1km ] time index {idt}')
    d1 = get_cross_data(ncfile_1km, idt, latLocs_1km, use_wrf_pblh=False)

    # ── matching 111 m time index (same ordering) ─────────────
    # idt_range starts at 138; we map 138→0, 139→1, …
    idx_111 = global_idt
    if idx_111 >= len(time_map):
        print(f'  No 111 m data for idt={idt}, skipping.')
        continue

    fi_111, t_111 = time_map[idx_111]

    # ── open 111 m smoothed file if needed ───────────────────
    if fi_111 != current_fi_sm:
        if ncfile_sm is not None:
            ncfile_sm.close()
        fp_sm      = baseFolder_sm + wrffi + tfiles[fi_111] + ':00:00'
        print(f'[smth] opening {fp_sm}')
        ncfile_sm  = Dataset(fp_sm)
        current_fi_sm = fi_111

    # ── open 111 m (no smooth) file if needed ────────────────
    if fi_111 != current_fi_111:
        if ncfile_111 is not None:
            ncfile_111.close()
        fp_111      = baseFolder_111 + wrffi + tfiles[fi_111] + ':00:00'
        print(f'[111m] opening {fp_111}')
        ncfile_111  = Dataset(fp_111)
        current_fi_111 = fi_111

    # ── 111 m smoothed data ───────────────────────────────────
    latLocs_sm = [xlim1, xlim2]
    print(f'[smth] time step {t_111}')
    d2 = get_cross_data(ncfile_sm, t_111, latLocs_sm, use_wrf_pblh=False)

    # ── 111 m data ────────────────────────────────────────────
    latLocs_111 = [xlim1, xlim2]
    print(f'[111m] time step {t_111}')
    d3 = get_cross_data(ncfile_111, t_111, latLocs_111, use_wrf_pblh=False)

    # ── transitions ───────────────────────────────────────────
    trans1 = find_transitions(d1['fr_profile'], d1['lats_np'][0, :], lat_limit=34.518)
    trans2 = find_transitions(d2['fr_profile'], d2['lats_np'][0, :], lat_limit=34.50)
    trans3 = find_transitions(d3['fr_profile'], d3['lats_np'][0, :], lat_limit=34.50)

    # ── time strings ──────────────────────────────────────────
    t1str = d1['tPDTstr'][9:13]   # HHMM
    t2str = d2['tPDTstr'][9:13]
    t3str = d3['tPDTstr'][9:13]

    # ── diagnostics ───────────────────────────────────────────
    for tag, d, tr in [('1km', d1, trans1), ('smth-1km', d2, trans2), ('111m', d3, trans3)]:
        print(f"  [{tag}] Fr {np.nanmin(d['fr_profile']):.2f}–{np.nanmax(d['fr_profile']):.2f}  "
              f"sub→super: {tr['sub_to_super']:.4f}" if tr['sub_to_super'] else
              f"  [{tag}] sub→super: not found")

    # ================================================================
    # ── FIGURE : 3 rows × 2 cols + 2 colorbar rows ──────────────────
    # ================================================================
    fig = plt.figure(figsize=(22, 18))

    gs = gridspec.GridSpec(
        5, 2,                                          # ← 5 rows instead of 4
        figure=fig,
        hspace=0.08,
        wspace=0.06,
        height_ratios=[1, 1, 1, 0.08, 0.06],          # ← extra spacer row (row 3)
    )
    
    ax_a   = fig.add_subplot(gs[0, 0])
    ax_b   = fig.add_subplot(gs[0, 1])
    ax_c   = fig.add_subplot(gs[1, 0])
    ax_d   = fig.add_subplot(gs[1, 1])
    ax_e   = fig.add_subplot(gs[2, 0])
    ax_f   = fig.add_subplot(gs[2, 1])
    ax_cbV = fig.add_subplot(gs[4, 0])               # ← row 4 instead of 3
    ax_cbW = fig.add_subplot(gs[4, 1])               # ← row 4 instead of 3
    # row 3 is the empty spacer — no subplot needed

    # ── row 1 : 1 km ─────────────────────────────────────────
    cf_V1 = draw_panel(ax_a,
                       d1['lats_cross'], d1['z_cross'], d1['V_cross'], d1['th_cross'],
                       d1['ter_line'], V_levels, tmap,
                       target_lat_1km, trans1,
                       show_ylabel=True, show_xlabel=False,
                       label_letter='a', time_str=f'1km {t1str}',
                       show_legend=True, show_fr_lines=True)

    cf_W1 = draw_panel(ax_b,
                       d1['lats_cross'], d1['z_cross'], d1['W_cross'], d1['th_cross'],
                       d1['ter_line'], W_levels, tmap,
                       target_lat_1km, trans1,
                       show_ylabel=False, show_xlabel=False,
                       label_letter='b', time_str=f'1km {t1str}',
                       show_legend=False, show_fr_lines=True)

    # ── row 2 : 111 m smoothed ───────────────────────────────
    draw_panel(ax_c,
               d2['lats_cross'], d2['z_cross'], d2['V_cross'], d2['th_cross'],
               d2['ter_line'], V_levels, tmap,
               target_lat_sm, trans2,
               show_ylabel=True, show_xlabel=False,
               label_letter='c', time_str=f'111m smth-1km {t2str}',
               show_legend=False, show_fr_lines=True)

    draw_panel(ax_d,
               d2['lats_cross'], d2['z_cross'], d2['W_cross'], d2['th_cross'],
               d2['ter_line'], W_levels, tmap,
               target_lat_sm, trans2,
               show_ylabel=False, show_xlabel=False,
               label_letter='d', time_str=f'111m smth-1km {t2str}',
               show_legend=False, show_fr_lines=True)

    # ── row 3 : 111 m ─────────────────────────────────────────
    draw_panel(ax_e,
               d3['lats_cross'], d3['z_cross'], d3['V_cross'], d3['th_cross'],
               d3['ter_line'], V_levels, tmap,
               target_lat_111, trans3,
               show_ylabel=True, show_xlabel=True,
               label_letter='e', time_str=f'111m {t3str}',
               show_legend=False, show_fr_lines=True)

    draw_panel(ax_f,
               d3['lats_cross'], d3['z_cross'], d3['W_cross'], d3['th_cross'],
               d3['ter_line'], W_levels, tmap,
               target_lat_111, trans3,
               show_ylabel=False, show_xlabel=True,
               label_letter='f', time_str=f'111m {t3str}',
               show_legend=False, show_fr_lines=True)

    # ── colorbars ─────────────────────────────────────────────
    # norm_V = plt.Normalize(vmin=-24, vmax=24)
    # sm_V   = plt.cm.ScalarMappable(cmap=tmap, norm=norm_V)
    # sm_V.set_array([])
    # cb_V   = plt.colorbar(sm_V, cax=ax_cbV, orientation='horizontal', extend='both')
    # cb_V.set_label('V-component (m s$^{-1}$)', fontsize=16)
    # cb_V.ax.tick_params(labelsize=14)
    # # tick every 4 m/s
    # cb_V.set_ticks(np.arange(-24, 25, 4))

    # norm_W = plt.Normalize(vmin=-5, vmax=5)
    # sm_W   = plt.cm.ScalarMappable(cmap=tmap, norm=norm_W)
    # sm_W.set_array([])
    # cb_W   = plt.colorbar(sm_W, cax=ax_cbW, orientation='horizontal', extend='both')
    # cb_W.set_label('W-component (m s$^{-1}$)', fontsize=16)
    # cb_W.ax.tick_params(labelsize=14)
    # cb_W.set_ticks(np.arange(-5, 6, 1))

    # ── V colorbar ───────────────────────────────────────────────
    norm_V = mcolors.BoundaryNorm(boundaries=V_levels, ncolors=tmap.N)
    sm_V   = plt.cm.ScalarMappable(cmap=tmap, norm=norm_V)
    sm_V.set_array([])
    cb_V   = plt.colorbar(sm_V, cax=ax_cbV, orientation='horizontal', extend='both')
    cb_V.set_label('V-component (m s$^{-1}$)', fontsize=18)
    cb_V.ax.tick_params(labelsize=14)
    cb_V.set_ticks(np.arange(-24, 25, 4))
    
    # ── W colorbar ───────────────────────────────────────────────
    norm_W = mcolors.BoundaryNorm(boundaries=W_levels, ncolors=tmap.N)
    sm_W   = plt.cm.ScalarMappable(cmap=tmap, norm=norm_W)
    sm_W.set_array([])
    cb_W   = plt.colorbar(sm_W, cax=ax_cbW, orientation='horizontal', extend='both')
    cb_W.set_label('W-component (m s$^{-1}$)', fontsize=18)
    cb_W.ax.tick_params(labelsize=14)
    cb_W.set_ticks(np.arange(-5, 6, 1))
    
    # ── save ──────────────────────────────────────────────────
    fName_out = (f'SWEX2022_IOP10_UWOW_Cross_Combined_thv1p5K_'
                 f'{d1["tPDTstr"]}_PDT.jpg')
    fig.savefig(fName_out, bbox_inches='tight', dpi=150)
    fig.clf()
    plt.close()
    print(f'  Saved {fName_out}')

    # ── cleanup ───────────────────────────────────────────────
    del d1, d2, d3, trans1, trans2, trans3

# ── close files ───────────────────────────────────────────────
ncfile_1km.close()
if ncfile_sm  is not None: ncfile_sm.close()
if ncfile_111 is not None: ncfile_111.close()

print('Done.')

Opening 1 km file …
[1km ] time index 138
[smth] opening /glade/campaign/univ/ucsb0023/duine/wrf/wrf460/sundowners/SWEX2022/IOP10/run_1km_545_24_z107_2way/ndown_111m_km_opt2_2Dom_smth1km/wrfout_d04_2022-05-12_17:00:00
[111m] opening /glade/campaign/univ/ucsb0023/duine/wrf/wrf460/sundowners/SWEX2022/IOP10/run_1km_545_24_z107_2way/ndown_111m_km_opt2_2Dom/wrfout_d04_2022-05-12_17:00:00
[smth] time step 0
[111m] time step 0
  [1km] sub→super: not found
  [smth-1km] sub→super: not found
  [111m] Fr 0.00–10.30  sub→super: 34.4822
  Saved SWEX2022_IOP10_UWOW_Cross_Combined_thv1p5K_20220512_1000_PDT.jpg
[1km ] time index 139
[smth] time step 1
[111m] time step 1
  [1km] sub→super: not found
  [smth-1km] Fr 0.01–5.23  sub→super: 34.4806
  [111m] Fr 0.00–16.68  sub→super: 34.4829
  Saved SWEX2022_IOP10_UWOW_Cross_Combined_thv1p5K_20220512_1010_PDT.jpg
[1km ] time index 140
[smth] time step 2
[111m] time step 2
  [1km] sub→super: not found
  [smth-1km] sub→super: not found
  [111m] Fr 0.01–5.32  